<a href="https://colab.research.google.com/github/Ikbal-ullah/JE-Early-Warning-System/blob/master/Phase4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# 1. Install Enterprise Geospatial Libraries
!pip install -q rasterio geopandas rasterstats shapely h3

import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import h3
from rasterstats import zonal_stats
import rasterio
from rasterio.windows import from_bounds
import os

print("--- INITIATING PHASE 4: OPTIMIZED DEMOGRAPHIC AND-GATE ---")

# 2. Download the UN-Adjusted Population Raster for India (WorldPop)
raster_url = "https://data.worldpop.org/GIS/Population/Global_2000_2020/2020/IND/ind_ppp_2020_UNadj.tif"
raster_path = "india_population.tif"

if not os.path.exists(raster_path):
    print("Downloading high-resolution human population raster from WorldPop...")
    !wget -nc -q -O {raster_path} {raster_url}
    print("Download secured.")
else:
    print("WorldPop raster already exists on disk. Bypassing download.")

# 3. Load your Phase 3 Biological Engine output
print("Loading Phase 3 Biological Matrix...")
df_bio = pd.read_csv("nalbari_phase3_biological_state.csv.gz")

# 4. Extract unique spatial zones and generate polygons
unique_hexagons = df_bio['hexagon'].unique()
print(f"Generating physical boundary polygons for {len(unique_hexagons)} vector zones...")

hex_polygons = []
for hex_id in unique_hexagons:
    boundary = h3.cell_to_boundary(hex_id)
    polygon = Polygon([(lon, lat) for lat, lon in boundary])
    hex_polygons.append({"hexagon": hex_id, "geometry": polygon})

gdf_hex = gpd.GeoDataFrame(hex_polygons, crs="EPSG:4326")

# 5. THE OPTIMIZATION: Bounding Box Raster Cropping
print("Cropping massive India raster down to the exact Nalbari bounding box...")
# Get the extreme outer edges of all our hexagons combined
minx, miny, maxx, maxy = gdf_hex.total_bounds

with rasterio.open(raster_path) as src:
    # Create a mathematical window that only covers Nalbari
    window = from_bounds(minx, miny, maxx, maxy, src.transform)
    # Extract only that specific window into RAM
    transform = src.window_transform(window)
    cropped_data = src.read(1, window=window)
    nodata_value = src.nodata

# 6. Execute Zonal Summation on the Cropped Array
print("Executing Zonal Summation on the cropped memory array (Should take < 10 seconds)...")
pop_stats = zonal_stats(gdf_hex, cropped_data, affine=transform, stats="sum", nodata=nodata_value)

gdf_hex['human_population'] = [stat['sum'] if stat['sum'] is not None else 0 for stat in pop_stats]

# 7. Merge and Apply the AND-Gate Logic
print("Merging demographics with biological state engine...")
df_final = pd.merge(df_bio, gdf_hex[['hexagon', 'human_population']], on='hexagon', how='left')

# The Spatial Filter: Suppress hazard if nobody lives there
df_final['human_spillover_hazard'] = df_final['rolling_14d_vidd'] * df_final['human_population']

# Clean and output
output_file = "nalbari_phase4_human_spillover.csv.gz"
df_final.to_csv(output_file, index=False, compression="gzip")

print(f"\nPhase 4 Complete. Mathematical Spatial Filter Applied.")
print(f"File saved: {output_file}")

from google.colab import files
files.download(output_file)

print("\n--- SAMPLE DEMOGRAPHIC OUTPUT ---")
print(df_final[['hexagon', 'human_population', 'rolling_14d_vidd', 'human_spillover_hazard']].drop_duplicates(subset=['hexagon']).head(15))

--- INITIATING PHASE 4: OPTIMIZED DEMOGRAPHIC AND-GATE ---
WorldPop raster already exists on disk. Bypassing download.
Loading Phase 3 Biological Matrix...
Generating physical boundary polygons for 2671 vector zones...
Cropping massive India raster down to the exact Nalbari bounding box...
Executing Zonal Summation on the cropped memory array (Should take < 10 seconds)...
Merging demographics with biological state engine...

Phase 4 Complete. Mathematical Spatial Filter Applied.
File saved: nalbari_phase4_human_spillover.csv.gz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


--- SAMPLE DEMOGRAPHIC OUTPUT ---
               hexagon  human_population  rolling_14d_vidd  \
0      883ce03401fffff        268.382660          3.532710   
1095   883ce03403fffff        429.255981          3.530843   
2190   883ce03405fffff         84.044037          3.524611   
3285   883ce03407fffff        153.178619          3.538095   
4380   883ce03409fffff        508.896362          3.542003   
5475   883ce0340bfffff        315.111023          3.523441   
6570   883ce0340dfffff        526.246460          3.535920   
7665   883ce03411fffff        401.907898          3.490668   
8760   883ce03413fffff        450.251282          3.492345   
9855   883ce03415fffff        480.027435          3.530323   
10950  883ce03417fffff        235.106384          3.531379   
12045  883ce03419fffff        414.377441          3.464236   
13140  883ce0341bfffff        496.478271          3.451635   
14235  883ce0341dfffff        309.682343          3.506223   
15330  883ce03429fffff        916.3